In [ ]:
# Dependencies
from typing import Callable, Dict, List, Optional, Tuple, Type, Union
from enum import Enum

import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patches as patches

import gymnasium
from gymnasium import Env
from gymnasium.spaces import Box, Discrete, Dict, Tuple
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy

import torch as th
import torch
from torch import nn

from sklearn.preprocessing import OneHotEncoder

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import copy

import pickle

# Environment

Observation space:
> $0$ — Ocean <br>
> $1$ — Island <br>
> $2$ — Submarine <br>
> $3$ — Travelled <br>

Action space:
> $0$ — UP <br>
> $1$ — DOWN <br>
> $2$ — LEFT <br>
> $3$ — RIGHT <br>
> $4$ — NOTHING <br>



In [ ]:
# Observation space
class Tile(Enum):
  OCEAN = 0
  SUBMARINE = 1
  ISLAND = 2
  TRAVELLED = 3

# Action space
class Action(Enum):
  UP = 0
  DOWN = 1
  LEFT = 2
  RIGHT = 3
  NOTHING = 4
  # RESURFACE = 5

In [ ]:
board = np.array(
    [
        [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],
        [0, 0, 2, 0, 0,  0, 2, 0, 0, 0,  0, 0, 2, 2, 0],
        [0, 0, 2, 0, 0,  0, 0, 0, 2, 0,  0, 0, 2, 0, 0],
        [0, 0, 0, 0, 0,  0, 0, 0, 2, 0,  0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],

        [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],
        [0, 2, 0, 2, 0,  0, 2, 0, 2, 0,  0, 0, 0, 0, 0],
        [0, 2, 0, 2, 0,  0, 2, 0, 0, 0,  0, 0, 0, 0, 0],
        [0, 0, 0, 2, 0,  0, 0, 2, 0, 0,  0, 2, 2, 2, 0],
        [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],

        [0, 0, 0, 2, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],
        [0, 0, 2, 0, 0,  0, 0, 2, 0, 0,  0, 2, 0, 0, 0],
        [2, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 2, 0, 0],
        [0, 0, 2, 0, 0,  0, 2, 0, 2, 0,  0, 0, 0, 2, 0],
        [0, 0, 0, 2, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],
    ]
)

print(board.shape)
print(board)

In [ ]:

class ActionLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.locations = []
        self.actions_log = []
        self.true_actions_log = []
        self.boards = []
        self.start_log = []
        self._temp = []
        self._location_temp = []
        self._true_temp = []
        self._start = []
        self.direction = [Action(num).name for num in range(len(Action))]

    def _on_step(self) -> bool:
        # Log the action taken
        done = self.locals["dones"][0]
        action = self.locals['actions'][0]
        true_action = self.locals["infos"][0]["true_action"]
        if type(self.locals["obs_tensor"]) == dict:
          location = self.locals["obs_tensor"]["Coordinates"]
          self._location_temp.append(location)

        self._temp.append(self.direction[action])
        self._true_temp.append(self.direction[true_action])
        if done == True:
          board = self.locals["infos"][0]["cumulative"]
          self.boards.append(board)
          self.actions_log.append(self._temp)
          self.true_actions_log.append(self._true_temp)
          if type(self.locals["obs_tensor"]) == dict:
            self.locations.append(self._location_temp)
          self._temp = []
          self._true_temp = []
          self._location_temp = []
        return True

In [ ]:
class ActionLoggerCallbackCNN(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.locations = []
        self.actions_log = []
        self.true_actions_log = []
        self.boards = []
        self.start_log = []
        self._temp = []
        self._location_temp = []
        self._true_temp = []
        self._start = []
        self.direction = [Action(num).name for num in range(len(Action))]

    def _on_step(self) -> bool:
        # Log the action taken
        done = self.locals["dones"][0]
        action = self.locals['actions'][0]
        true_action = self.locals["infos"][0]["true_action"]

        self._temp.append(self.direction[action])
        self._true_temp.append(self.direction[true_action])
        if done == True:
          board = self.locals["infos"][0]["cumulative"]
          self.boards.append(board)
          self.actions_log.append(self._temp)
          self.true_actions_log.append(self._true_temp)
          self._temp = []
          self._true_temp = []
          self._location_temp = []
        return True

In [ ]:
def RandomAction(env, episodes = 1000):
  scores = []
  for episode in range(1, episodes + 1):
    state = env.reset()
    done = False
    score = 0

    while not done:
      action = env.action_space.sample()
      n_state, reward, done, truncated, info = env.step(action)
      score += reward

    scores.append(score)

    # print(f"Episode: {episode}, Score: {score}")
  print(f"Mean score from {episodes} episodes: {sum(scores) / episodes}")

In [ ]:
def CycleModel(model, callback, className, cycles = 100):
  mean, std = [], []

  for _ in range(cycles):
    print(f"Number {_} of {cycles}")
    model.learn(total_timesteps=2048, callback=callback)
    model.get_env().reset()
    m, s = evaluate_policy(model, model.get_env(), n_eval_episodes=25, deterministic=True)
    print(f"Deterministic mean: {m} and std: {s}")
    mean.append(m)
    std.append(s)

  mean = np.array(mean)
  std = np.array(std)
  ax.plot(range(cycles), mean, label=className)
  ax.fill_between(range(cycles), mean - std, mean+std, alpha=0.2)

In [ ]:
fig, ax = plt.subplots()


In [ ]:
def arrow_and_heatmap(model, env):
  map_explored = []
  
  final_boards = [
    [], [], [],
    [], [], [],
    [], [], [],
  ]
  final_directions = [
      [], [], [],
      [], [], [],
      [], [], [],
  ]
  grid_size = env.grid_size
  for x in range(grid_size):
    for y in range(grid_size):
      env.reset()
      if env.board[y, x] == 0:
        agent = np.array([y, x])
        obs, info = env.reset(manual_spawn=agent, random_damage=False)

        done = False
        while done == False:
          action, _ = model.predict(obs, deterministic=True)
          obs, reward, done, truncated, info = env.step(int(action))

      area_number = (y // 5) * 3 + (x // 5) + 1
      final_boards[area_number - 1].append(env.board)
      final_directions[area_number - 1].append(env.directions)

  for section in range(9):
      vectorized = np.array([[[env._vectorize_action[x] for x in row] for row in matrix] for matrix in final_directions[section]])
      print(vectorized.shape)
      vectorized_avg = np.mean(vectorized, axis=0).reshape(15*15, 2)
      y = vectorized_avg[..., 0:1] * -1
      x = vectorized_avg[..., 1:2]

      map_explored.append((((x == 0) & (y == 0)).sum() - 31) / (15*15 - 31))

  print(map_explored)
  print(np.mean(map_explored))

  vectorized = np.array([[[[env._vectorize_action[x] for x in row] for row in matrix] for matrix in section] for section in final_directions])
  vectorized_avg = np.mean(vectorized, axis=1).reshape(9, 15*15, 2)
  y = vectorized_avg[..., 0:1] * -1
  x = vectorized_avg[..., 1:2]

  magnitude =  np.sqrt(x**2 + y**2)
  x_normalized = x / (magnitude + 1e-9)
  y_normalized = y / (magnitude + 1e-9)

  fig, ax = plt.subplots(3, 3, figsize=(9, 9))

  fig.suptitle("Average vectorized direction of the starting positions within the red box")

  for section in range(9):
    x = section // 3
    y = section % 3

    # Replace numbers smaller than 2 with 0
    filtered_matrices = [np.where(matrix == 2, 0, matrix) for matrix in final_boards[section]]

    # Sum all filtered matrices
    result = np.sum(filtered_matrices, axis=0) / 3
    ax[x, y].imshow(result, cmap='viridis', interpolation='nearest', aspect="auto")

    X, Y = np.meshgrid(range(15), range(15))

    ax[x, y].quiver(
        X,
        Y,
        x_normalized[section],
        y_normalized[section],
        color="white",
        scale_units="width",
        scale=20
    )

    rect = patches.Rectangle((-0.5, -0.5), 15, 15, linewidth=1, edgecolor='white', facecolor='none')
    ax[x, y].add_patch(rect)
    rect = patches.Rectangle((y * 5 - 0.5, x* 5 - 0.5), 5, 5, linewidth=1, edgecolor='red', facecolor='none')
    ax[x, y].add_patch(rect)

    plt.subplots_adjust(wspace=0, hspace=0)
    for a in ax.flat:
      a.axis('off')

  plt.show()

  fig_overall, ax_overall = plt.subplots(1, 1, figsize=(9, 9))

  vectorized_overall_avg = np.mean(np.mean(vectorized, axis=1), axis=0).reshape(15*15, 2)
  y = vectorized_overall_avg[..., 0:1] * -1
  x = vectorized_overall_avg[..., 1:2]

  magnitude =  np.sqrt(x**2 + y**2)
  x_normalized = x / (magnitude + 1e-9)
  y_normalized = y / (magnitude + 1e-9)

  # Replace numbers smaller than 2 with 0
  filtered_matrices = [[np.where(matrix == 2, 0, matrix) for matrix in section] for section in final_boards]

  # Sum all filtered matrices
  result = np.sum(np.sum(filtered_matrices, axis=0), axis=0) / 3
  ax_overall.imshow(result, cmap='viridis', interpolation='nearest', aspect="auto")

  X, Y = np.meshgrid(range(15), range(15))

  ax_overall.quiver(
      X,
      Y,
      x_normalized,
      y_normalized,
      color="white",
  )

  ax_overall.axis('off')

  plt.show()



# Simple Captain Sonar
Instead of whole board, only agent location is passed through

In [ ]:
class SimpleCaptainSonarEnv(Env):

  metadata = {"render_modes": ["human"], "render_fps": 30}

  def __init__(self, board, max_length = 50):
    super().__init__()
    self.agent               = np.array([0, 0], dtype=int)
    self.repairable_movement = np.array([0] * len(Action), dtype=int)
    self.permanent_movement  = np.array([0] * len(Action), dtype=int)
    self._board              = board.copy()
    self.board               = board.copy()
    self.cumulative          = board.copy()
    self.grid_size           = board.shape[0]
    self.length              = 0
    self.max_length          = max_length
    self.travelled           = []
    self.true_actions        = []
    self.update_travelled()
    self.directions          = np.full(board.shape, Action.NOTHING.value)

    """
    What can the RL agent see?
    > Agent coordinates: [x, y]
    — Range: [0, 14]

    > Repairable movement: [N, S, E, W]
    — Range: [0, 2]

    > Permanent damage movement: [N, S, E, W]
    — Range: [0, 2]
    """
    self.observation_space = Dict({
        # Agent coordinates: [x, y]
        "Coordinates": Box(low=0, high=self.grid_size, shape=(2,), dtype=int),
        # Repairable movement: [N, S, E, W]
        "Repairable_movement": Box(low=0, high=len(Action)-1, shape=(len(Action),), dtype=int),
        # Permanent damage movement: [N, S, E, W]
        "Permanent_movement": Box(low=0, high=len(Action)-1, shape=(len(Action),), dtype=int),
        # Action masking: prevents illigal moves
        "action_mask": Box(low=0, high=1, shape=(len(Action),), dtype=int)
    })

    """
    What can agent do?
    > Move UP
    > Move DOWN
    > Move LEFT
    > Move RIGHT
    > Move Nothing
    5 actions in total
    """
    self.action_space = Discrete(len(Action))

    # Each action vectorized
    self._vectorize_action = {
        # Going from row 1 to 0 is UP
        Action.UP.value:      np.array([ -1 ,  0 ]),
        # Going from row 0 to 1 is DOWN
        Action.DOWN.value:    np.array([  1 ,  0 ]),
        # Going from column 1 to 0 is LEFT
        Action.LEFT.value:    np.array([  0 , -1 ]),
        # Going from column 0 to 1 is RIGHT
        Action.RIGHT.value:   np.array([  0 ,  1 ]),
        Action.NOTHING.value: np.array([  0 ,  0 ]),
    }

  def step(self, action):
    """
    RESURFACE action skips all movement related steps
    """
    done = False
    truncated = False
    # if action == Action.RESURFACE.value:
    #   observation_space = self.get_observation_space()
    #   info = {"true_action": action}
    #   reward = -5
    #   done = False
    #   truncated = False

    #   # Reset movement trackers
    #   self.repairable_movement = np.array([0] * len(Action))
    #   self.permanent_movement  = np.array([0] * len(Action))

    #   # Remove previous been points on board
    #   self.board = self._board.copy()

    #   # Update length
    #   self.length += 1

    #   return observation_space, reward, done, truncated, info


    """
    First, change the agent's action based on the algorithm's answer
    Give 1 reward as the longer the agent survives, the better
    """
    x_previous = self.agent[0]
    y_previous = self.agent[1]
    self.board[x_previous, y_previous] = Tile.TRAVELLED.value
    self.cumulative[x_previous, y_previous] += Tile.TRAVELLED.value
    vectorized_movement = self._vectorize_action[action]
    self.agent += vectorized_movement
    reward = 1

    """
    Second, if agent is outside boundaries, in an island or already been there,
    which is not allowed, then reverse action and punish.
    Otherwise, note where agent has been on the board
    """
    true_action = action
    x = self.agent[0]
    y = self.agent[1]
    self.directions[x_previous, y_previous] = true_action
    if x == -1 or y == -1 or x == self.grid_size or y == self.grid_size or self.board[x, y] == Tile.TRAVELLED.value or self.board[x, y] == Tile.ISLAND.value:
      self.agent -= vectorized_movement
      true_action = Action.NOTHING.value
    else:
      self.board[x, y] = Tile.SUBMARINE.value

    if true_action == Action.NOTHING.value:
      reward -= (self.repairable_movement[true_action] + 1)

    """
    Third, update repairable movement in the correct direction.
    If repairable movement is already 3,
    then add to permanent damage movement
    """
    if true_action == Action.NOTHING.value or self.repairable_movement[true_action] < 3:
      self.repairable_movement[true_action] += 1

    elif true_action != Action.NOTHING.value:
      self.permanent_movement[true_action] += 1

    """
    Fourth, add repair logic.
    Check if repairable_movement NORTH, SOUTH or WEST has 3 and EAST is at least 1,
    then substract for NORTH, SOUTH or WEST 3 and 1 from EAST
    """
    if true_action == Action.RIGHT.value and np.any(self.repairable_movement == 3):
      action_to_be_repaired = np.where(self.repairable_movement == 3)[0][0]

      # Only combination of (NORTH, SOUTH, WEST) x (EAST) is repairable
      if action_to_be_repaired != Action.RIGHT.value:
        # reward += 5
        self.repairable_movement[Action.RIGHT.value] -= 1
        self.repairable_movement[action_to_be_repaired] -= 3

    if true_action != Action.NOTHING.value and true_action != Action.RIGHT.value and self.repairable_movement[true_action] == 3 and self.repairable_movement[Action.RIGHT.value] > 0:
      # reward += 5
      self.repairable_movement[Action.RIGHT.value] -= 1
      self.repairable_movement[true_action] -= 3

    """
    Fifth, if repairable movement and permanent damage add up to 6 in any direction,
    then simulation is done and punish
    """
    if self.repairable_movement[true_action] + self.permanent_movement[true_action] >= 6:
      done = True
      # reward -= 30

    """
    Sixth, if simulation has been running for X steps,
    then simulation is done and reward with X points
    """
    self.length += 1
    if self.length >= self.max_length:
      done = True
      truncated = True

    """
    Seventh, translate self.agent, self.repairable_movement
    and self.permanent_movement into oberservation_space
    """
    observation_space = self.get_observation_space()
    info = {"true_action": true_action}
    self.update_travelled()
    self.true_actions.append(true_action)

    if done == True:
      info["board"] = self.board
      info["cumulative"] = self.cumulative
      info["travelled"] = self.travelled
      info["directions"] = self.directions
      info["true_actions"] = self.true_actions
    return observation_space, int(reward), done, truncated, info

  def render(self):
    pass

  def update_travelled(self):
    x = self.agent[0]
    y = self.agent[1]
    self.travelled.append(np.array([copy.deepcopy(x), copy.deepcopy(y)]))

  def reset(self, manual_spawn=False, random_damage=False, seed=None, options=None):
    super().reset(seed=seed)
    self.np_random = np.random.RandomState(seed)

    # Remove previous been points on board
    self.board = self._board.copy()
    self.cumulative = self._board.copy()
    self.directions = np.full(board.shape, Action.NOTHING.value)
    self.true_actions = []

    # Reset the length
    self.length = 0

    # Initialize random starting position of submarine
    if type(manual_spawn) == bool:
      self.agent = self.np_random.randint(0, self.grid_size, 2)
      while self.board[self.agent[1], self.agent[0]] == Tile.ISLAND.value:
        self.agent = self.np_random.randint(0, self.grid_size, 2)
    else:
      self.agent = manual_spawn
    x = self.agent[0]
    y = self.agent[1]
    self.board[x, y] = Tile.SUBMARINE.value
    self.travelled = []
    self.update_travelled()

    # Reset movement trackers
    if random_damage:
      self.repairable_movement = self.np_random.randint(0, 3 + 1, 4)
      self.permanent_movement  = self.np_random.randint(0, 3 + 1, 4)
    else:
      self.repairable_movement = np.array([0] * len(Action))
      self.permanent_movement  = np.array([0] * len(Action))

    # Return observation
    return (self.get_observation_space(), {})

  def get_observation_space(self):
    action_mask = []
    current_position = self.agent.copy()
    for a in Action:
      vectorized_movement = self._vectorize_action[a.value]
      future_position = current_position + vectorized_movement
      x = future_position[0]
      y = future_position[1]
      if x == -1 or y == -1 or x >= self.grid_size or y >= self.grid_size or self.board[x, y] == Tile.TRAVELLED.value or self.board[x, y] == Tile.ISLAND.value:
        # 0 is Invalid
        action_mask.append(0)
      else:
        # 1 is Valid
        action_mask.append(1)

    return {
        # Agent coordinates: [x, y]
        "Coordinates": self.agent,
        # Repairable movement: [N, S, E, W]
        "Repairable_movement": self.repairable_movement,
        # Permanent damage movement: [N, S, E, W]
        "Permanent_movement": self.permanent_movement,
        # Action mask
        "action_mask": np.array(action_mask)
    }


In [ ]:
max_length = 100
trials = 400

In [ ]:
simple_env = SimpleCaptainSonarEnv(board, max_length)
check_env(simple_env)
RandomAction(simple_env)

In [ ]:
callback_simple = ActionLoggerCallback()
simple_model = PPO("MultiInputPolicy", simple_env, verbose=1, learning_rate=1e-4)
CycleModel(simple_model, callback_simple, "Simple", trials)

In [ ]:
fig

In [ ]:
arrow_and_heatmap(simple_model, simple_env)

In [ ]:
# Save model weights
torch.save(simple_model.policy.state_dict(), "simple_weights.pth")

# Intermediate Captain Sonar
Input is the whole board, but linearly put in

In [ ]:
class IntermediateCaptainSonarEnv(SimpleCaptainSonarEnv):
  def __init__(self, board, max_length = 50):
    super().__init__(board, max_length)
    self.observation_space["Coordinates"] = Box(low=0, high=1, shape=(self.grid_size ** 2 * len(Tile),), dtype=int)
    self.encoder = OneHotEncoder(sparse_output=False, categories=[range(len(Tile))])

  def get_observation_space(self):
    observation_space = super().get_observation_space()
    encoded_board = self.board.reshape(-1, 1)
    encoded_board = self.encoder.fit_transform(encoded_board).flatten()
    encoded_board = encoded_board.astype(np.int64)
    observation_space["Coordinates"] = encoded_board
    return observation_space


In [ ]:
intermediate_env = IntermediateCaptainSonarEnv(board, max_length)
check_env(intermediate_env)
RandomAction(intermediate_env)

In [ ]:
callback_intermediate = ActionLoggerCallback()
intermediate_model = PPO("MultiInputPolicy", intermediate_env, verbose=1, learning_rate=1e-4)
CycleModel(intermediate_model, callback_intermediate, "Intermediate", trials)

In [ ]:
fig

In [ ]:
arrow_and_heatmap(intermediate_model, intermediate_env)

In [ ]:
# Save model weights
torch.save(intermediate_model.policy.state_dict(), "intermediate_weights.pth")

# Complex Captain Sonar

Input is the whole board with a 2D matrix. Using CNN policy

In [ ]:
class CustomCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gymnasium.spaces.Dict, linear_dim = 32, feature_extract = 32):
        # We do not know features-dim here before going over all the items,
        # so put something dummy for now. PyTorch requires calling
        # nn.Module.__init__ before adding modules
        super().__init__(observation_space, features_dim=1)

        extractors = {}

        total_concat_size = 0
        # We need to know size of the output of this extractor,
        # so go over all the spaces and compute output feature sizes
        for key, subspace in observation_space.spaces.items():
            if "movement" in key or key == "action_mask":
                # These are discrete
                features = subspace.shape[0]
                extractors[key] = nn.Sequential(
                    nn.Linear(features, linear_dim),
                    nn.ReLU(),
                    nn.Linear(linear_dim, linear_dim),
                    nn.ReLU(),
                    nn.Linear(linear_dim, linear_dim),
                    nn.ReLU(),
                )
                total_concat_size += linear_dim
            elif key == "Coordinates":
                # Run through CNN
                n_input_channels = subspace.shape[0]
                extractors[key] = nn.Sequential(
                  nn.Conv2d(n_input_channels, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.MaxPool2d(2, 2),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.MaxPool2d(2, 2),
                  nn.Flatten(),
                )
                total_concat_size += subspace.shape[1] // 2 // 2 * feature_extract + subspace.shape[1] // 2 // 2 * feature_extract

        self.extractors = nn.ModuleDict(extractors)

        # Update the features dim manually
        self._features_dim = 320

    def forward(self, observations) -> th.Tensor:
        encoded_tensor_list = []

        # self.extractors contain nn.Modules that do all the processing.
        for key, extractor in self.extractors.items():
            encoded_tensor_list.append(extractor(observations[key]))
        # Return a (B, self._features_dim) PyTorch tensor, where B is batch dimension.
        return th.cat(encoded_tensor_list, dim=1)

policy_kwargs_CNN = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(linear_dim=32, feature_extract=32),
)

In [ ]:
class ComplexCaptainSonarEnv(SimpleCaptainSonarEnv):
  def __init__(self, board, max_length = 50):
    super().__init__(board, max_length)

    self.board = self.board.astype(np.uint8)
    self.observation_space = Dict({
        "Coordinates": Box(low=0, high=1, shape=(len(Tile), self.grid_size, self.grid_size), dtype=np.uint8),
        "action_mask": self.observation_space["action_mask"]
    })
    self.encoder = OneHotEncoder(sparse_output=False, categories=[range(len(Tile))])

  def get_observation_space(self):
    _observation_space = super().get_observation_space()
    observation_space = {}
    encoded_board = self.board.reshape(-1, 1)
    encoded_board = self.encoder.fit_transform(encoded_board).T
    encoded_board = encoded_board.astype(np.int64)
    observation_space["Coordinates"] = encoded_board.reshape(len(Tile), self.grid_size, self.grid_size)
    observation_space["action_mask"] = _observation_space["action_mask"]
    return observation_space

In [ ]:
complex_env = ComplexCaptainSonarEnv(board, max_length)
RandomAction(complex_env)

In [ ]:
callback_complex = ActionLoggerCallback()
complex_model = PPO("MultiInputPolicy", complex_env, verbose=1, policy_kwargs=policy_kwargs_CNN, learning_rate=1e-4)
CycleModel(complex_model, callback_complex, "Complex", trials)

In [ ]:
fig

In [ ]:
arrow_and_heatmap(complex_model, complex_env)

In [ ]:
# Save model weights
torch.save(complex_model.policy.state_dict(), "complex_weights.pth")

# Hybrid Complex Captain Sonar

Input is the whole board with a 2D matrix. Using custom policy to accept both CNN and linear inputs

In [ ]:
class CustomHybrid(BaseFeaturesExtractor):
    def __init__(self, observation_space: gymnasium.spaces.Dict, linear_dim = 32, feature_extract = 32):
        # We do not know features-dim here before going over all the items,
        # so put something dummy for now. PyTorch requires calling
        # nn.Module.__init__ before adding modules
        super().__init__(observation_space, features_dim=1)

        extractors = {}

        total_concat_size = 0
        # We need to know size of the output of this extractor,
        # so go over all the spaces and compute output feature sizes
        for key, subspace in observation_space.spaces.items():
            if "movement" in key or key == "action_mask":
                # These are discrete
                features = subspace.shape[0]
                extractors[key] = nn.Sequential(
                    nn.Linear(features, linear_dim),
                    nn.ReLU(),
                    nn.Linear(linear_dim, linear_dim),
                    nn.ReLU(),
                    nn.Linear(linear_dim, linear_dim),
                    nn.ReLU(),
                )
                total_concat_size += linear_dim
            elif key == "Coordinates":
                # Run through CNN
                n_input_channels = subspace.shape[0]
                extractors[key] = nn.Sequential(
                  nn.Conv2d(n_input_channels, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.MaxPool2d(2, 2),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.MaxPool2d(2, 2),
                  nn.Flatten(),
                )
                total_concat_size += subspace.shape[1] // 2 // 2 * feature_extract + subspace.shape[1] // 2 // 2 * feature_extract

        self.extractors = nn.ModuleDict(extractors)

        # Update the features dim manually
        self._features_dim = 384

    def forward(self, observations) -> th.Tensor:
        encoded_tensor_list = []

        # self.extractors contain nn.Modules that do all the processing.
        for key, extractor in self.extractors.items():
            encoded_tensor_list.append(extractor(observations[key]))
        # Return a (B, self._features_dim) PyTorch tensor, where B is batch dimension.
        return th.cat(encoded_tensor_list, dim=1)

policy_kwargs_hybrid = dict(
    features_extractor_class=CustomHybrid,
    features_extractor_kwargs=dict(linear_dim=32, feature_extract=32),
)

In [ ]:
class HyrbidCaptainSonarEnv(SimpleCaptainSonarEnv):
  def __init__(self, board, max_length = 50):
    super().__init__(board, max_length)

    self.board = self.board.astype(np.uint8)
    self.observation_space["Coordinates"] = Box(low=0, high=len(Tile), shape=(len(Tile), self.grid_size, self.grid_size), dtype=np.uint8)
    self.encoder = OneHotEncoder(sparse_output=False, categories=[range(len(Tile))])

  def get_observation_space(self):
    observation_space = super().get_observation_space()
    encoded_board = self.board.reshape(-1, 1)
    encoded_board = self.encoder.fit_transform(encoded_board).T
    encoded_board = encoded_board.astype(np.uint8)
    observation_space["Coordinates"] = encoded_board.reshape(len(Tile), self.grid_size, self.grid_size)
    return observation_space

In [ ]:
hybrid_env = HyrbidCaptainSonarEnv(board, max_length)
check_env(hybrid_env)
RandomAction(hybrid_env)

In [ ]:
callback_hybrid = ActionLoggerCallbackCNN()
hybrid_model = PPO("MultiInputPolicy", hybrid_env, verbose=1, policy_kwargs=policy_kwargs_hybrid, learning_rate=1e-4)
CycleModel(hybrid_model, callback_hybrid, "Hybrid", trials)

In [ ]:
fig

In [ ]:
arrow_and_heatmap(hybrid_model, hybrid_env)

In [ ]:
torch.save(hybrid_model.policy.state_dict(), "hybrid_weights.pth")

In [ ]:
fig.savefig('Training_score.png', dpi=300, bbox_inches='tight')

# Generate data for Agent 2 and Agent 4

In [ ]:
def generate_data(env, model, deterministic = True, trials = 10, grid_size=15):
  sequences_movement = []
  final_paths = []


  for number in range(trials):
    print(f"Processing {number + 1}")
    for x in range(grid_size):
      for y in range(grid_size):
        env.reset()
        if env.board[y, x] == 0:
          agent = np.array([y, x])
          obs, info = env.reset(manual_spawn=agent, random_damage=False)

          done = False
          actions = ""
          while done == False:
            action, _ = model.predict(obs, deterministic=deterministic)
            actions = actions + f" {action}"
            obs, reward, done, truncated, info = env.step(int(action))
          sequences_movement.append(actions)
          final_paths.append(env.travelled)
    print(f"Processed {number + 1} of {trials}")

  return sequences_movement, final_paths

In [ ]:
# Agent 2
callback_intermediate = ActionLoggerCallback()
loaded_intermediate_model = PPO("MultiInputPolicy", intermediate_env, verbose=1, learning_rate=1e-4)
loaded_intermediate_model.policy.load_state_dict(torch.load("intermediate_weights.pth"))

intermediate_sequences, intermediate_paths = generate_data(intermediate_env, loaded_intermediate_model, deterministic = True, trials = 100, grid_size=15)

In [ ]:
with open("intermediate_sequences.pkl", "wb") as f:
    pickle.dump(intermediate_sequences, f)

with open("intermediate_paths.pkl", "wb") as f:
    pickle.dump(intermediate_paths, f)

In [ ]:
# Agent 4
callback_hybrid = ActionLoggerCallbackCNN()
loaded_hybrid_model = PPO("MultiInputPolicy", hybrid_env, verbose=1, policy_kwargs=policy_kwargs_hybrid, learning_rate=1e-4)
loaded_hybrid_model.policy.load_state_dict(torch.load("hybrid_weights.pth"))

hybrid_sequences, hybrid_paths = generate_data(hybrid_env, loaded_hybrid_model, deterministic = True, trials = 100, grid_size=15)

In [ ]:
with open("hybrid_sequences_test.pkl", "wb") as f:
    pickle.dump(hybrid_sequences, f)

with open("hybrid_paths.pkl", "wb") as f:
    pickle.dump(hybrid_paths, f)